# Emergency Vision AI: Production Person-Crop Training, Evaluation & GPU Benchmark

This notebook provides the unified, reproducible Google Colab GPU workflow for **Emergency Vision AI**.

### Architecture & Single Source of Truth
- **GitHub**: Sole source of truth for all code, scripts, configurations, and canonical notebook.
- **Google Drive**: Persistent authoritative dataset storage (`/content/drive/MyDrive/emergency-vision-ai/data/urfd`) and backup destination for trained models and benchmark results.
- **Google Colab**: Ephemeral GPU execution runtime (Tesla T4) that bootstraps itself idempotently.
- **Second-Stage Action Training**: Aligns training with production inference by extracting 16-frame person-crop tubes (`YOLO11n` + `ByteTrack` + 5% padding) with strict sequence-level isolation (Seed=42).
- **Workflow**: **Bootstrap → Train → Evaluate → Benchmark → Summary**.

## 1. Bootstrap Environment & Google Drive

Clones or synchronizes the repository from GitHub, mounts Google Drive, verifies the authoritative dataset (`30 FALL + 40 NORMAL` videos), materializes Git LFS models, installs production dependencies, creates the local dataset symlink, and generates a complete environment status report.

In [4]:
# ==============================================================================
# 1. BOOTSTRAP ENVIRONMENT & GOOGLE DRIVE
# ==============================================================================
import os
REPO_DIR = "/content/emergency-vision-ai"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mukhammadiev01-1/emergency-vision-ai.git {REPO_DIR}

%cd {REPO_DIR}

import scripts.colab_bootstrap as bootstrap
bootstrap.run_bootstrap()


Cloning into '/content/emergency-vision-ai'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 313 (delta 119), reused 260 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (313/313), 26.33 MiB | 7.07 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Filtering content: 100% (2/2), 131.96 MiB | 2.63 MiB/s, done.
/content/emergency-vision-ai
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

      EMERGENCY VISION AI — COLAB ENVIRONMENT REPORT
Environment Mode:      Google Colab
CUDA Accelerator:      Tesla T4 (Available: True)
GPU Memory:            14.56 GB
Repository Root:       /content/e

{'timestamp': '2026-09-05T10:43:10.791172+00:00',
 'is_colab': True,
 'cuda': {'cuda_available': True,
  'device_name': 'Tesla T4',
  'gpu_count': 1,
  'total_memory_gb': 14.56,
  'torch_version': '2.11.0+cu128'},
 'repo': {'repo_dir': '/content/emergency-vision-ai',
  'commit_sha': 'ae12c8d567f713691c5fea29c83e34f68f4a4029',
  'branch': 'main'},
 'dataset': {'dataset_path': '/content/drive/MyDrive/emergency-vision-ai/data/urfd',
  'fall_count': 30,
  'normal_count': 40,
  'total_count': 70,
  'intact': True},
 'models': {'action_checkpoint': {'path': '/content/emergency-vision-ai/models/action_recognition/r3d18_urfd_best.pth',
   'size_mb': 126.6,
   'sha256': '52cc51fd016263e7529009f23147d7a91b8855d685f11239346016ff55eadb5c',
   'valid': True},
  'yolo_checkpoint': {'path': '/content/emergency-vision-ai/models/detection/yolo11n.pt',
   'size_mb': 5.35,
   'sha256': '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1',
   'valid': True}},
 'versions': {'torch': '2.11.0+c

## 2. Second-Stage Person-Crop Action Training

Trains/fine-tunes R3D-18 on production-style 16-frame person crops generated via YOLO11n + ByteTrack with hard-negative mining and strict sequence isolation (Seed=42: 49 train, 10 val, 11 test).

Saves best model to `models/action_recognition/r3d18_urfd_person_crops.pth` and automatically creates a persistent backup in Google Drive.

In [5]:
# ==============================================================================
# 2. SECOND-STAGE PERSON-CROP ACTION TRAINING (ONE COMMAND)
# ==============================================================================
!python scripts/train_person_crop_pipeline.py \
    --dataset-root data/urfd \
    --base-checkpoint models/action_recognition/r3d18_urfd_best.pth \
    --yolo-model models/detection/yolo11n.pt \
    --output-dir models/action_recognition \
    --checkpoint-name r3d18_urfd_person_crops.pth \
    --epochs 12 \
    --batch-size 8 \
    --lr 1e-4 \
    --device cuda \
    --seed 42 \
    --drive-backup-dir /content/drive/MyDrive/emergency-vision-ai/models/action_recognition


Выходные данные были обрезаны до нескольких последних строк (5000).

0: 256x640 1 person, 7.8ms
Speed: 0.7ms preprocess, 7.8ms inference, 1.3ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 7.2ms
Speed: 0.8ms preprocess, 7.2ms inference, 1.3ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 7.6ms
Speed: 0.7ms preprocess, 7.6ms inference, 1.3ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 8.4ms
Speed: 0.7ms preprocess, 8.4ms inference, 1.2ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 7.1ms
Speed: 0.8ms preprocess, 7.1ms inference, 1.3ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 12.7ms
Speed: 0.8ms preprocess, 12.7ms inference, 1.7ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 7.6ms
Speed: 0.8ms preprocess, 7.6ms inference, 1.3ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 1 person, 7.4ms
Speed: 0.8ms preprocess, 7.4ms inf

## 3. Production Pipeline Comparative Evaluation

Evaluates both checkpoints side-by-side on the **actual production pipeline** (YOLO11n → ByteTrack → Person Crop → R3D-18 → Temporal Confirmation) across `fall-01..05` and `adl-01..05`.

Generates a side-by-side comparison of max $P(\text{FALL})$, confirmed emergency events, recall, and false positive rates.

In [6]:
# ==============================================================================
# 3. PRODUCTION PIPELINE COMPARATIVE EVALUATION
# ==============================================================================
!python scripts/evaluate_production_pipeline.py \
    --action-model models/action_recognition/r3d18_urfd_person_crops.pth \
    --compare-with models/action_recognition/r3d18_urfd_best.pth \
    --dataset-root data/urfd \
    --max-fall-videos 5 \
    --max-normal-videos 5 \
    --device cuda \
    --output-json results/eval/pipeline_comparison.json


      EMERGENCY VISION AI — PRODUCTION PIPELINE COMPARATIVE EVALUATION
Model A (Target / Evaluated): models/action_recognition/r3d18_urfd_person_crops.pth
Model B (Reference / Baseline): models/action_recognition/r3d18_urfd_best.pth
2026-09-05 11:03:58,655 [INFO] >>> Evaluating Model A: models/action_recognition/r3d18_urfd_person_crops.pth
EMERGENCY VISION AI — PRODUCTION PIPELINE EVALUATION
Hardware Accelerator:      Tesla T4 (CUDA)
Action Model Checkpoint:   models/action_recognition/r3d18_urfd_person_crops.pth (Exists: True)
YOLO Detection Weights:    models/detection/yolo11n.pt (Exists: True)
Decision Threshold:        0.70
Consecutive Required:      2 windows
Inference Interval:        8 frames
Cooldown Period:           5.0 s
Crop Padding Ratio:        5%
Discovered 10 videos to evaluate:
  - FALL Videos:   5
  - NORMAL Videos: 5
2026-09-05 11:04:00,380 [INFO] Loading R3D-18 action recognition checkpoint from models/action_recognition/r3d18_urfd_person_crops.pth on cuda...
2026-0

## 4. Production Pipeline GPU Benchmark (Tesla T4)

Measures end-to-end detection latency, person crop preprocessing, R3D-18 inference latency, and overall pipeline throughput (FPS) on the GPU accelerator.

In [7]:
# ==============================================================================
# 4. PRODUCTION PIPELINE GPU BENCHMARK (Tesla T4)
# ==============================================================================
!python scripts/benchmark_gpu.py \
    --video data/urfd/videos/fall/fall-01-cam0.mp4 \
    --action-model models/action_recognition/r3d18_urfd_person_crops.pth \
    --yolo-model models/detection/yolo11n.pt \
    --device cuda \
    --threshold 0.70 \
    --interval 8 \
    --warmup 5 \
    --output-json results/benchmark_gpu_results.json


2026-09-05 11:05:57,246 [INFO] Hardware Accelerator: Tesla T4 (CUDA)
2026-09-05 11:05:57,247 [INFO] Target Video:        data/urfd/videos/fall/fall-01-cam0.mp4
2026-09-05 11:05:57,247 [INFO] Action Checkpoint:   models/action_recognition/r3d18_urfd_person_crops.pth (Exists: True)
2026-09-05 11:05:57,247 [INFO] YOLO Model:          models/detection/yolo11n.pt
2026-09-05 11:05:58,699 [INFO] Loading R3D-18 action recognition checkpoint from models/action_recognition/r3d18_urfd_person_crops.pth on cuda...
2026-09-05 11:05:59,926 [INFO] R3D-18 action recognition model ready on device: cuda
2026-09-05 11:05:59,927 [INFO] Running 5 warm-up iterations on CUDA...
2026-09-05 11:06:01,283 [INFO] Warm-up completed.
2026-09-05 11:06:01,296 [INFO] Executing benchmark over 160 frames...
2026-09-05 11:06:03,668 [WARNING] >>> CONFIRMED EMERGENCY ACTION on canonical_gpu_benchmark [Track ID 1]: FALL (Confidence: 1.00, Hits: 2, Latency: 31.07ms)

       EMERGENCY VISION AI: PRODUCTION PIPELINE BENCHMARK R

## 5. Final Artifacts & Google Drive Summary

Verifies the saved model checkpoints and structured JSON reports in both local workspace and Google Drive.

In [8]:
# ==============================================================================
# 5. FINAL ARTIFACTS & GOOGLE DRIVE SUMMARY
# ==============================================================================
import glob
import os

print("=" * 80)
print("      TRAINING & BENCHMARK ARTIFACTS SUMMARY")
print("=" * 80)
checkpoints = [
    "models/action_recognition/r3d18_urfd_best.pth",
    "models/action_recognition/r3d18_urfd_person_crops.pth",
    "/content/drive/MyDrive/emergency-vision-ai/models/action_recognition/r3d18_urfd_person_crops.pth",
]
print("Model Checkpoints:")
for ckpt in checkpoints:
    if os.path.exists(ckpt):
        sz = os.path.getsize(ckpt) / (1024 * 1024)
        print(f"  • {ckpt} ({sz:.2f} MB)")
    else:
        print(f"  • {ckpt} [NOT FOUND]")

print("\nGenerated Results Reports:")
for r in sorted(glob.glob("results/**/*.json", recursive=True)):
    print(f"  • {r}")
for r in sorted(glob.glob("/content/drive/MyDrive/emergency-vision-ai/results/**/*.json", recursive=True)):
    print(f"  • {r} (Drive Backup)")
print("=" * 80)


      TRAINING & BENCHMARK ARTIFACTS SUMMARY
Model Checkpoints:
  • models/action_recognition/r3d18_urfd_best.pth (126.60 MB)
  • models/action_recognition/r3d18_urfd_person_crops.pth (126.60 MB)
  • /content/drive/MyDrive/emergency-vision-ai/models/action_recognition/r3d18_urfd_person_crops.pth (126.60 MB)

Generated Results Reports:
  • results/benchmark_gpu_results.json
  • results/eval/pipeline_comparison.json
  • results/training/train_person_crops_results.json
  • /content/drive/MyDrive/emergency-vision-ai/results/training/train_person_crops_results.json (Drive Backup)
  • /content/drive/MyDrive/emergency-vision-ai/results/urfd_r3d18/evaluation_metrics.json (Drive Backup)
  • /content/drive/MyDrive/emergency-vision-ai/results/urfd_r3d18/experiment_config.json (Drive Backup)


In [9]:
from pathlib import Path
import shutil

repo = Path("/content/emergency-vision-ai")
drive_root = Path("/content/drive/MyDrive/emergency-vision-ai")

files_to_backup = [
    repo / "results/eval/pipeline_comparison.json",
    repo / "results/benchmark_gpu_results.json",
    repo / "models/action_recognition/r3d18_urfd_person_crops_metadata.json",
]

for src in files_to_backup:
    if src.exists():
        rel = src.relative_to(repo)
        dst = drive_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"BACKED UP: {rel}")
    else:
        print(f"NOT FOUND: {src}")

BACKED UP: results/eval/pipeline_comparison.json
BACKED UP: results/benchmark_gpu_results.json
BACKED UP: models/action_recognition/r3d18_urfd_person_crops_metadata.json
